In [390]:
import ast
import json
import re
import requests

import pandas as pd

from bs4 import BeautifulSoup

pd.set_option('display.max_columns', 500)

url = 'https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC' # Mapa de la Comunidad de Madrid y alrededores
df = pd.read_csv('../data/pisos_madrid.csv', sep ='|')

# Extracción de datos

In [391]:
def extraer_informacion(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    estate_component = soup.find("estate-show-v2")
    if not estate_component:
        return []
    
    estate_json_raw = estate_component.get(":estate")
    estate_json_raw = estate_json_raw.replace("&quot;", '"')
    estate_data = json.loads(estate_json_raw)

    data = {
        'dormitorios': estate_data.get('rooms'),
        'superficie_m2': estate_data.get('numeric_surface'),
        'baños': estate_data.get('bathrooms'),
        'url': estate_data.get('detail_url'),
        'features': estate_data.get('features'),
        'descripcion': estate_data.get('description'),
        'precio': estate_data.get('costs'),
        'latitud': estate_data.get('latitude'),
        'longitud': estate_data.get('longitude'),
        'media': estate_data.get('media'),
        'points_of_interest': estate_data.get('points_of_interest'),
        'energy_data': estate_data.get('energy_data'),
    }
    
    return data

In [392]:
def obtener_urls(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    print(f'Buscando pisos en la página {url} ...')
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")
    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')
    estates_data = json.loads(estates_raw)

    data = []

    for estate in estates_data:
        url_piso = estate.get("detail_url")

        if url_piso in df['url'].to_list():
            print('Ya lo tengo:', url_piso)
            return data
        data.append(extraer_informacion(url_piso))

    return data

In [393]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
ultima = soup.find('a', string='>>')
max_pages = int(re.findall(r'pag-(\d+)', str(ultima))[0])

url_splited = url.split('pag-1')
data = []
for i in range(1, max_pages+1):
    subdata = obtener_urls(f'pag-{i}'.join(url_splited))
    data.extend(subdata)

    if len(subdata) < 15:
        break

df = pd.concat([pd.DataFrame(data), df])
df.to_csv('../data/pisos_madrid.csv', sep='|', index=False)

Buscando pisos en la página https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC ...
Ya lo tengo: https://www.tecnocasa.es/venta/piso/guadalajara/yebes--valdeluz/647894.html


# Pretratamiento de datos

In [394]:
# A partir de este punto sería interesante integrar un pipeline, para que si entra un nuevo piso todo se ejecute en el orden adecuado.

In [395]:
df[['features', 'precio', 'media', 'points_of_interest', 'energy_data']] = df[['features', 'precio', 'media', 'points_of_interest', 'energy_data']].fillna('{}')

df['planta'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('floor'))
df['aire_acondicionado'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('air_conditioning'))
df['ascensor'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('elevator'))
df['calefaccion'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('heating'))
df['categoria'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('category'))
df['año_construccion'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('build_year'))

df['precio'] = df['precio'].replace(' €', '', regex=True)
df['precio_euros'] = df['precio'].apply(ast.literal_eval).apply(lambda x: x.get('price'))
df['hipoteca'] = df['precio'].apply(ast.literal_eval).apply(lambda x: x.get('mortgage_payment_value'))

df['planos'] = df['media'].apply(ast.literal_eval).apply(lambda x: x.get('floor_plans') != None)
df['realistico'] = df['media'].apply(ast.literal_eval).apply(lambda x: x.get('has_realistico'))
df['fotografias'] = df['media'].apply(ast.literal_eval).apply(lambda x: len(x.get('images')))

df['transporte_publico'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('public_transport'))
df['escuelas'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('school'))
df['farmacias'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('pharmacy'))
df['hospitales'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('hospital'))
df['supermercados'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('market'))
df['tiendas'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('shop'))
df['bares'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('bar'))
df['restaurantes'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('restaurant'))

df['clase_energetica'] = df['energy_data'].apply(ast.literal_eval).apply(lambda x: x.get('class_emissions'))
df['eficiencia_energetica'] = df['energy_data'].apply(ast.literal_eval).apply(lambda x: x.get('efficiency'))
df['emisiones_energeticas'] = df['energy_data'].apply(ast.literal_eval).apply(lambda x: x.get('emissions'))

In [396]:
# Aquí extraemos la información de las columnas de puntos de interes: Total de elementos por columna y distancia del más cercano.

In [397]:
# La columna de descripción podría tener información valiosa si le aplicamos un LLM (HuggingFace)

In [398]:
df

,dormitorios,superficie_m2,baños,url,features,descripcion,precio,latitud,longitud,media,points_of_interest,energy_data,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,transporte_publico,escuelas,farmacias,hospitales,supermercados,tiendas,bares,restaurantes,clase_energetica,eficiencia_energetica,emisiones_energeticas
0,2 dorm.,89.0,2 baños,https://www.tecnocasa.es/venta/piso/guadalajar...,"{'id': 647894, 'floor': 1, 'floors': None, 'bo...",<p>Agencia inmobiliaria de Yebes zona VALDELUZ...,"{'price': '216.000', 'box_price': None, 'mortg...",40.591102,-3.110843,"{'floor_plans': None, 'has_realistico': True, ...",{'public_transport': [{'name': 'Guadalajara/Ye...,"{'class': 'e', 'class_emissions': 'f', 'certif...",1,NaN,Sí,Independiente,Popular,2008,216.000,696,False,True,26,"[{'name': 'Guadalajara/Yebes', 'class': 'railw...","[{'name': 'CEIP Luz de Yebes', 'class': 'schoo...","[{'name': 'Farmacia', 'class': 'pharmacy', 'su...","[{'name': 'Sanatorio de Alcohete', 'class': 'h...","[{'name': 'Simply', 'class': 'grocery', 'subcl...","[{'name': 'Tiendas', 'class': 'shop', 'subclas...","[{'name': 'Capri', 'class': 'bar', 'subclass':...","[{'name': 'Tramonti', 'class': 'restaurant', '...",f,94 kW h/m&sup2; año,117
1,NaN,90.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 649701, 'floor': 2, 'floors': None, 'bo...","<p data-start=""90"" data-end=""251"">TECNOCASA GU...","{'price': '569.000', 'box_price': None, 'mortg...",40.433602,-3.672523,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'Diego de León'...,"{'class': 'e', 'class_emissions': 'e', 'certif...",2,Independiente (Frío / Calor),,Independiente (Gas),Media,1965,569.000,1.833,False,False,20,"[{'name': 'Diego de León', 'class': 'railway',...","[{'name': 'CEPA Joaquín Sorolla', 'class': 'sc...","[{'name': 'Farmacia - Calle Alonso Heredia 5',...",[{'name': 'Instituto Provincial de Rehabilitac...,"[{'name': 'La Plaza de DIA', 'class': 'grocery...","[{'name': 'Panadarío', 'class': 'shop', 'subcl...","[{'name': 'Cervecería Bar Los Calamares', 'cla...","[{'name': 'La Buena Mesa', 'class': 'restauran...",e,284 kW h/m&sup2; año,"73,2"
2,2 dorm.,65.0,2 baños,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 643217, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de Madrid - Chamberí -...,"{'price': '299.000', 'box_price': None, 'mortg...",40.441002,-3.696503,"{'floor_plans': [{'id': 9485692, 'width': 1024...","{'public_transport': [{'name': 'Alonso Cano', ...","{'class': 'e', 'class_emissions': 'e', 'certif...",Baja,NaN,,Independiente (Gas),Media,1950,299.000,963,True,False,22,"[{'name': 'Alonso Cano', 'class': 'railway', '...","[{'name': 'Liceo Italiano', 'class': 'school',...","[{'name': 'Farmacia - Calle Ríos Rosas 50', 'c...","[{'name': 'Hospitales', 'class': 'hospital', '...","[{'name': 'Dia Market', 'class': 'grocery', 's...","[{'name': 'Tiendas', 'class': 'shop', 'subclas...","[{'name': 'Rick’s', 'class': 'bar', 'subclass'...","[{'name': 'Il Pastaio del Vecchio Mulino', 'cl...",e,"227,1 kWh/m&sup2; año","47,3"
3,3 dorm.,72.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mor...,"{'id': 645163, 'floor': 2, 'floors': None, 'bo...","<p data-start=""189"" data-end=""411"">La agencia ...","{'price': '220.000', 'box_price': None, 'mortg...",40.677702,-3.966313,"{'floor_plans': [{'id': 9539314, 'width': 1154...",{'public_transport': [{'name': 'Estación de Au...,"{'class': 'e', 'class_emissions': 'e', 'certif...",2,NaN,,Independiente (Eléctrica),Popular,1976,220.000,709,True,False,24,[{'name': 'Estación de Autobuses de Moralzarza...,"[{'name': 'Moralzarzal (Esc. Mun. Música)', 'c...","[{'name': 'Farmacia de Pereda', 'class': 'phar...",None,"[{'name': 'Supersol', 'class': 'grocery', 'sub...","[{'name': 'Alimentación Bazar', 'class': 'shop...","[{'name': 'Cava del Coso', 'class': 'bar', 'su...","[{'name': 'El Fogón de los Arrieros', 'class':..

In [399]:
df = df.drop(columns=['url', 'features', 'descripcion', 'precio', 'media', 'points_of_interest', 'energy_data', 'transporte_publico', 'escuelas', 'farmacias', 'hospitales', 'supermercados', 'tiendas', 'bares', 'restaurantes'])

# Limpieza de datos

In [400]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

imputer.fit_transform(pd.DataFrame(df['dormitorios'].replace(' dorm.', '', regex=True)))

array([[2.],
       [3.],
       [2.],
       ...,
       [3.],
       [3.],
       [3.]], shape=(1206, 1))

In [401]:
df['emisiones_energeticas'].replace(',', '.')

0         117
1        73,2
2        47,3
3          61
4          81
        ...  
1201     22.8
1202    10,24
1203       35
1204       51
1205      NaN
Name: emisiones_energeticas, Length: 1206, dtype: str

In [388]:
df['dormitorios'] = df['dormitorios'].replace(' dorm.', '', regex=True)
df['baños'] = df['baños'].replace(' baño[s]*', '', regex=True)
df['planta'] = df['planta'].replace(r'\s*\(.*\)', '', regex=True)
df['aire_acondicionado'] = df['aire_acondicionado'].replace(r'\s+.+', '', regex=True)
df['calefaccion_gas'] = df['calefaccion'].str.contains('gas', case=False)
df['calefaccion_electrica'] = df['calefaccion'].str.contains('eléctrica', case=False)
df['calefaccion'] = df['calefaccion'].replace(r'\s+.+', '', regex=True)
df['eficiencia_energetica'] = df['eficiencia_energetica'].replace(r'\s.*', '', regex=True).replace(',', '.', regex=True)
df['emisiones_energeticas'] = df['emisiones_energeticas'].replace(',', '.', regex=True)

In [389]:
df

,dormitorios,superficie_m2,baños,latitud,longitud,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,clase_energetica,eficiencia_energetica,emisiones_energeticas,calefaccion_gas,calefaccion_electrica
0,2,89.0,2,40.591102,-3.110843,1,NaN,Sí,Independiente,Popular,2008,216.000,696,False,True,26,f,94,117,False,False
1,NaN,90.0,1,40.433602,-3.672523,2,Independiente,,Independiente,Media,1965,569.000,1.833,False,False,20,e,284,73.2,True,False
2,2,65.0,2,40.441002,-3.696503,Baja,NaN,,Independiente,Media,1950,299.000,963,True,False,22,e,227.1,47.3,True,False
3,3,72.0,1,40.677702,-3.966313,2,NaN,,Independiente,Popular,1976,220.000,709,True,False,24,e,291,61,False,True
4,2,76.0,1,40.406300,-3.737990,6,NaN,,NaN,Media,1961,180.000,580,False,False,19,g,407,81,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1201,3,100.0,2,40.365902,-3.698932,,Independiente,,Independiente,Popular,1994,300.000,966,False,False,25,d,112,22.8,True,False
1202,1,72.0,1,40.549202,-3.324873,Baja,NaN,,Independiente,Media,1989,129.500,417,False,False,18,b,60.26,10.24,False,False
1203,NaN,54.0,1,40.428102,-3.638793,,NaN,,NaN,Media,1970,225.000,725,False,False,11,e,210,35,False,False
1204,NaN,66.0,1,40.429402,-3.638183,2,NaN,,NaN,Popular,1965,270.000,870,False,False,16,e,152,51,False,False


In [402]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1206 entries, 0 to 1205
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dormitorios            1096 non-null   str    
 1   superficie_m2          1206 non-null   float64
 2   baños                  1110 non-null   str    
 3   latitud                1206 non-null   float64
 4   longitud               1206 non-null   float64
 5   planta                 1206 non-null   object 
 6   aire_acondicionado     450 non-null    str    
 7   ascensor               1206 non-null   str    
 8   calefaccion            669 non-null    str    
 9   categoria              1206 non-null   str    
 10  año_construccion       1206 non-null   int64  
 11  precio_euros           1206 non-null   str    
 12  hipoteca               1206 non-null   str    
 13  planos                 1206 non-null   bool   
 14  realistico             1206 non-null   bool   
 15  fotografias    

In [404]:
# Primero haremos un clustering con latitud/longitud para agrupar las viviendas por los núcleos poblacionales.

In [403]:
# Una vez tengamos nuestro modelo, hacer modelos con menos features, valorando la mejora de rendimiento/peso del modelo, frente a la perdida de precisión.